# What is PL/SQL?

### PL/SQL = Procedural Language / Structured Query Language
### Its Oracle progrramming language that extends SQL by adding proggramming features such as variables, conditions, loops, exception handelling, procedures, functions, packages, and triggers.

# Connection Establishment :

## Path :

In [23]:
path = r'E:\Study\Github\Sec\Scripts\DB Loading\connect_db.py'
exec(open(path, encoding='utf-8').read())

## Connection :

In [34]:
connect_oracle()

✨ Permanent %%plsql registered from secrets file! Session is live.


## Checking : 

In [26]:
%%plsql
DECLARE
    v_msg VARCHAR2(100) := 'Brilliant! Your database setup is officially complete.';
BEGIN
    DBMS_OUTPUT.PUT_LINE(v_msg);
END;

Brilliant! Your database setup is officially complete.


# GH

In [29]:
connect_oracle()

✨ Permanent %%plsql registered from secrets file! Session is live.


In [31]:
%%plsql
DECLARE
    v_user    VARCHAR2(100);
    v_schema  VARCHAR2(100);
BEGIN
    -- 1. Grab your connection identities safely
    SELECT USER, SYS_CONTEXT('userenv', 'current_schema')
    INTO v_user, v_schema
    FROM dual;

    -- 2. Print verification info directly using native PL/SQL buffer
    DBMS_OUTPUT.PUT_LINE('🟢 CONNECTION VERIFIED SUCCESSFULLY!');
    DBMS_OUTPUT.PUT_LINE('-------------------------------------------');
    DBMS_OUTPUT.PUT_LINE('👤 Logged-in User : ' || v_user);
    DBMS_OUTPUT.PUT_LINE('📂 Current Schema : ' || v_schema);
    DBMS_OUTPUT.PUT_LINE('-------------------------------------------');
EXCEPTION
    WHEN OTHERS THEN
        DBMS_OUTPUT.PUT_LINE('Error reading metadata: ' || SQLERRM);
END;


🟢 CONNECTION VERIFIED SUCCESSFULLY!
-------------------------------------------
👤 Logged-in User : CON_V_I_C_TSDG_SCHEMA_BOHNN
📂 Current Schema : CON_V_I_C_TSDG_SCHEMA_BOHNN
-------------------------------------------


In [32]:
%%plsql
DECLARE
    TYPE t_tables IS TABLE OF VARCHAR2(100);
    v_tables t_tables := t_tables(
        'JOB_HISTORY', 'EMPLOYEES', 'DEPARTMENTS', 'JOBS', 'LOCATIONS', 
        'COUNTRIES', 'REGIONS', 'ORDER_ITEMS', 'ORDERS', 'SHIPMENTS', 
        'INVENTORY', 'STORES', 'CUSTOMERS', 'COSTS', 'SALES', 
        'PROMOTIONS', 'PRODUCTS', 'CHANNELS', 'SUPPLEMENTARY_DEMOGRAPHICS', 'TIMES'
    );
BEGIN
    DBMS_OUTPUT.PUT_LINE('🧹 STARTING SCHEMA CLEANUP...');
    DBMS_OUTPUT.PUT_LINE('-------------------------------------------');
    
    FOR i IN 1..v_tables.COUNT LOOP
        BEGIN
            -- Drops tables locally since you are inside your own schema space
            EXECUTE IMMEDIATE 'DROP TABLE ' || v_tables(i) || ' CASCADE CONSTRAINTS';
            DBMS_OUTPUT.PUT_LINE('🗑️ Successfully Dropped: ' || v_tables(i));
        EXCEPTION
            WHEN OTHERS THEN
                -- Catch ORA-00942 (Table does not exist) and ignore it cleanly
                IF SQLCODE = -942 THEN
                    DBMS_OUTPUT.PUT_LINE('⚪ Skipped (Not found): ' || v_tables(i));
                ELSE
                    DBMS_OUTPUT.PUT_LINE('❌ Error dropping ' || v_tables(i) || ': ' || SQLERRM);
                END IF;
        END;
    END LOOP;
    DBMS_OUTPUT.PUT_LINE('-------------------------------------------');
    DBMS_OUTPUT.PUT_LINE('✨ All specified targets have been processed!');
END;


🧹 STARTING SCHEMA CLEANUP...
-------------------------------------------
⚪ Skipped (Not found): JOB_HISTORY
⚪ Skipped (Not found): EMPLOYEES
⚪ Skipped (Not found): DEPARTMENTS
⚪ Skipped (Not found): JOBS
⚪ Skipped (Not found): LOCATIONS
⚪ Skipped (Not found): COUNTRIES
⚪ Skipped (Not found): REGIONS
⚪ Skipped (Not found): ORDER_ITEMS
⚪ Skipped (Not found): ORDERS
⚪ Skipped (Not found): SHIPMENTS
⚪ Skipped (Not found): INVENTORY
⚪ Skipped (Not found): STORES
⚪ Skipped (Not found): CUSTOMERS
⚪ Skipped (Not found): COSTS
⚪ Skipped (Not found): SALES
⚪ Skipped (Not found): PROMOTIONS
⚪ Skipped (Not found): PRODUCTS
⚪ Skipped (Not found): CHANNELS
⚪ Skipped (Not found): SUPPLEMENTARY_DEMOGRAPHICS
⚪ Skipped (Not found): TIMES
-------------------------------------------
✨ All specified targets have been processed!


In [33]:
connect_oracle_sql()

The sql extension is already loaded. To reload it, use:
  %reload_ext sql
⚡ Oracle %%sql engine initialized successfully under alias: oracle_db


In [10]:
%%sql
SELECT table_name 
FROM user_tables 
ORDER BY table_name ASC;


Running query in 'oracle+oracledb://'

table_name


In [ ]:
import os
import re
from IPython import get_ipython


def execute_sql_file_statements(file_path, cursor):
    """Reads a local SQL file, strips environmental tags, and separates individual queries safely."""
    if not os.path.exists(file_path):
        print(f"⚠️ Target file not found: {file_path}")
        return 0, 0

    with open(file_path, "r", encoding="utf-8", errors="ignore") as f:
        raw_content = f.read()

    # Clean out environment configurations (like SET DEFINE OFF, COMMIT, prompt text)
    cleaned = re.sub(r'(?i)SET\s+\w+\s+\w+;', '', raw_content)
    cleaned = re.sub(r'(?i)COMMIT;', '', cleaned)
    # Strip Oracle remark comments
    cleaned = re.sub(r'(?i)REM\s+.*', '', cleaned)

    # Split independent commands by semicolon
    statements = cleaned.split(";")

    success_count = 0
    fail_count = 0

    for statement in statements:
        sql = statement.strip()
        # Skip empty lines, SQL*Plus prompt lines, or wrapper calls
        if not sql or sql.upper().startswith("PROMPT") or sql.upper().startswith("@@"):
            continue

        try:
            cursor.execute(sql)
            success_count += 1
        except Exception as e:
            fail_count += 1

    return success_count, fail_count


def deploy_local_sample_schemas():
    """Iterates through your local extracted repository, building and populating schemas in order."""
    base_dir = r"E:\Study\Github\Repositories\Learn\SQL\PL SQL\PL SQL By Prashant\Sample Data\db-sample-schemas-23.3"

    # 1. Proactively pull the live DB engine state directly from your setup environment
    ip = get_ipython()
    if "oracle_engine" not in ip.user_ns:
        print("❌ Error: Active Oracle engine connection state not found. Please run connect_oracle() first.")
        return

    # Extract the native engine instance and establish a connection bridge
    engine = ip.user_ns["oracle_engine"]
    connection = engine.raw_connection()
    cursor = connection.cursor()

    # Enforces dependency order: Base configuration data first, transaction modules last
    execution_plan = [
        {"folder": "human_resources", "create": "hr_create.sql",
            "populate": "hr_populate.sql", "name": "Human Resources (HR)"},
        {"folder": "customer_orders", "create": "co_create.sql",
            "populate": "co_populate.sql", "name": "Customer Orders (CO)"},
        {"folder": "sales_history",   "create": "sh_create.sql",
            "populate": "sh_populate.sql", "name": "Sales History (SH)"}
    ]

    print("Example 23c Sample Data Deployment Initiated...")
    print("=================================================================")

    for module in execution_plan:
        module_path = os.path.join(base_dir, module["folder"])
        print(f"\n📦 Processing Module: {module['name']}...")

        # Step 1: Execute Table Layout Structures (DDL)
        create_file = os.path.join(module_path, module["create"])
        ok_c, err_c = execute_sql_file_statements(create_file, cursor)
        print(f"   🔨 Tables Created/Configured: {ok_c} structural statements.")

        # Step 2: Stream Data Record Insert Matrices (DML)
        populate_file = os.path.join(module_path, module["populate"])
        ok_p, err_p = execute_sql_file_statements(populate_file, cursor)
        print(f"   📥 Records Successfully Populated: {ok_p} rows.")
        if err_p > 0:
            print(f"   ⚪ Structural Skips / Handled Gracefully: {err_p}")

        # Commit the transaction block for this specific module
        connection.commit()

    # Close local workspace streaming links safely
    cursor.close()
    connection.close()

    print("\n=================================================================")
    print("✨ SUCCESS: All local tables and rows deployed to your user schema!")


# Run the full schema pipeline execution
deploy_local_sample_schemas()

Example 23c Sample Data Deployment Initiated...

📦 Processing Module: Human Resources (HR)...


In [14]:
%%sql
SELECT table_name, 
       TO_NUMBER(EXTRACTVALUE(xmltype(dbms_xmlgen.getxml('SELECT COUNT(*) c FROM ' || table_name)), '/ROWSET/ROW/C')) AS current_rows
FROM user_tables
ORDER BY current_rows DESC;


Running query in 'oracle+oracledb://'

table_name,current_rows
